# 当前SOXX Goal 候选准入与依赖审计

只读已封存合同、Qlib账户结论、固定信息门和来源卡；范围限定九条命名路线。不读取新行情、拟合或运行账户。

In [1]:
import datetime as dt,hashlib,json
from pathlib import Path
root=Path.cwd();card=root/'research/stage1_soxx_goal_candidate_admission_audit_20261024';plan_path=card/'plan.json';plan=json.loads(plan_path.read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('Qlib_object_reads','new_market_GETs','new_model_fits','new_native_accounts','new_bootstraps','new_Recorders'))
obj={}
for name,(relative,digest) in plan['sources'].items():
 p=root/relative;assert sha(p)==digest,name
 if p.suffix=='.json':obj[name]=json.loads(p.read_text())
base=obj['exact_f696_baseline_result'];bound=obj['exact_f696_cash_window_bound'];dedup=obj['common_return_dedup_0bp'];bridge=obj['joint_SOXX_bridge_0bp'];info=obj['joint_SOXX_bridge_information'];binary=obj['original39_binary'];late=obj['late_residual_joint_value'];short=obj['soxl_ret3_rank'];cboe=obj['Cboe_variance_information'];ibkr=obj['IBKR_option_source'];aum=obj['SOXL_AUM_source'];sched=obj['Cboe_forward_scheduler']
assert abs(base['full']['relative_wealth']-.6860064369263443)<1e-12
assert bound['original_positive_windows']==255 and bound['rolling126_windows']==819 and bound['required_positive_windows_70pct']==574
assert dedup['candidate_SOXX_relative_wealth']<1 and dedup['rolling126_positive']<dedup['rolling126_required']
assert not bridge['historical_0bp_gate_pass'] and info['decision'].startswith('conditional_')
assert binary['information_pass'] is False and late['decision'].startswith('not_promoted_') and short['decision'].startswith('not_promoted_')
assert cboe['decision']=='not_promoted_information_gate_failed' and ibkr['decision']=='source_entitlement_passed_feature_not_qualified'
assert aum['historical_daily_AUM_shares_qualified'] is False and len(sched['missed_windows'])==3
routes=[
 {'route':'exact_f696_A0','stage':'frozen_complete_0bp_reference','status':'failed_SOXX_hard_gate','evidence':'exact_f696_baseline_result','ready_for_new_account':False,'reason':'Existing reference wealth and rolling windows fail; it is the parent, not a new candidate.'},
 {'route':'common_return_dedup_C','stage':'own_state_0bp_completed','status':dedup['decision'],'evidence':'common_return_dedup_0bp','ready_for_new_account':False,'reason':'Frozen version fails SOXX wealth, complete2025, rolling126 and paired lower gates; cannot resume by mask/seed tuning.'},
 {'route':'direct_joint_SOXX_margin_to_account','stage':'conditional_information_then_own_state_0bp','status':bridge['decision'],'evidence':'joint_SOXX_bridge_0bp','ready_for_new_account':False,'reason':'Information passed its narrow old-head control, but fixed mapping never triggered and exact f696 account equals the failing parent.'},
 {'route':'original39_direct_SOXL_cash_binary','stage':'information_screen','status':binary['decision'],'evidence':'original39_binary','ready_for_new_account':False,'reason':'Only1/6 fixed information gates passed; 2025/2026 sign ranking failed.'},
 {'route':'prior_close_late_signed_residual_joint_value','stage':'conditional_source_and_information_screen','status':late['decision'],'evidence':'late_residual_joint_value','ready_for_new_account':False,'reason':'Only3/6 local-value gates passed; historical minute first-live receipt remains unqualified.'},
 {'route':'existing_SOXL_ret3_contrarian','stage':'one_factor_necessary_rank','status':short['decision'],'evidence':'soxl_ret3_rank','ready_for_new_account':False,'reason':'2023 AUC<0.5 and 39-selection-corrected block lower<0; original39 already contains the coordinate.'},
 {'route':'Cboe_total_options_activity','stage':'source_forward_ongoing_and_fixed_information_closed','status':cboe['decision'],'evidence':'Cboe_variance_information','ready_for_new_account':False,'reason':'Five-date vintages are source-only and fixed variance information gate failed; no strategy forward day.'},
 {'route':'SOXX_option_skew_IBKR','stage':'current_quote_entitlement_only','status':ibkr['decision'],'evidence':'IBKR_option_source','ready_for_new_account':False,'reason':'No qualified2011-2022 strike/IV/Delta history or original quote-time receipts.'},
 {'route':'SOXL_actual_rebalance_demand','stage':'official_source_gap','status':aum['decision'],'evidence':'SOXL_AUM_source','ready_for_new_account':False,'reason':'No qualified2011-2022 daily AUM/shares and first-public vintages; current product page/scripts only.'}
]
assert not any(z['ready_for_new_account'] for z in routes)
result={'status':'scoped_no_current_candidate_ready_for_new_exact_f696_0bp_account','audited_at_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
        'plan_sha256':sha(plan_path),'scope_limit':plan['scope'],'baseline':{'relative_wealth':base['full']['relative_wealth'],'positive_rolling126':bound['original_positive_windows'],'rolling126_total':819,'rolling126_required':574,'cash_opportunity_log':base['full']['cash_opportunity_log'],'SOXL_selection_log':base['full']['SOXL_selection_log']},
        'closest_completed_0bp':{'route':'common_return_dedup_C','relative_wealth':dedup['candidate_SOXX_relative_wealth'],'rolling126_positive':dedup['rolling126_positive'],'rolling126_required':dedup['rolling126_required'],'complete_years':dedup['complete_years']},
        'routes':routes,'named_routes_checked':len(routes),'ready_for_new_exact_f696_0bp':0,'Cboe_source_missed_windows':len(sched['missed_windows']),
        'qualified_strategy_forward_days_in_named_routes':0,'historical_passing_candidate_in_named_routes':False,
        'next_independent_work':['Let frozen Cboe source-only heartbeat observe future official vintages in its actual UTC windows; these are not strategy days.','Seek a genuinely different predecision mechanism with source identity, first availability and an economic receiver that can alter both f696 allocation intent and own-state risk capacity.'],
        'user_constraint':'No supplier outreach now, per direct user instruction; no purchase/subscription.',
        'new_market_GETs':0,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'new_Recorders':0,
        'SOXX_goal_achieved':False,'limits':['This audit covers only explicitly named current routes, not all possible future factors or sparse-checkout branches.','Historical development failures do not prove any future mechanism impossible.','Source-only forward vintages and idle/scheduled work do not count as qualified economic forward days.']}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'routes':len(routes),'ready':0,'baseline_wealth':result['baseline']['relative_wealth'],'closest_wealth':result['closest_completed_0bp']['relative_wealth']})


{'status': 'scoped_no_current_candidate_ready_for_new_exact_f696_0bp_account', 'routes': 9, 'ready': 0, 'baseline_wealth': 0.6860064369263443, 'closest_wealth': 0.9614726395822718}


## 中文结论

# SOXX Goal 当前九条命名路线：没有可准入的新0bp候选

限定读取合同v2、精确f696原A_0原生Qlib账户、最接近的完整候选账户、近期固定信息/必要门及三类来源依赖；每份sidecar按计划SHA验证。**这只覆盖`result.json`列出的九条当前命名路线，不是稀疏工作区所有可能因子的穷尽证明。** 本卡0新行情、Qlib对象、拟合、bootstrap或账户。

精确f696 0bp Tiered父本的SOXX相对财富 **0.686006**，仅 **255/819** 个126日窗口胜SOXX，合同至少需要 **574/819**；现金机会对数 **−0.60032**、SOXL相对选择 **+0.22593**。最接近的已完成公共收益去重C账户，财富 **0.96147**、正窗 **369/819**，完整2025年相对财富 **0.93747**，固定版仍未过财富、年度、滚动及不确定性门，不能换掩码/种子继续救援。直接联合SOXX边际信息只对较弱旧头有条件增量，其精确f696自身状态0bp映射 **零触发**，账户仍等于失败父本。

近期原39项直接SOXL/现金二分类仅过1/6信息门；尾盘残差联合含费局部价值3/6；既有三日SOXL反转必要排序3/5，均在账户前关闭。Cboe总期权活跃度的固定方差信息门失败，后续五日观察仅为来源；IBKR当日期权报价没有合格2011–2022历史链；Direxion静态页/内嵌数据/产品chunk均未给出历史日级SOXL AUM与首发版本。用户已表示暂不联系供应商，未发起询价或购买。

决定 `scoped_no_current_candidate_ready_for_new_exact_f696_0bp_account`：不把测试通过、来源运输、局部标签或已失败版当作经济晋升依据。下一独立工作是让已修正时钟的Cboe来源心跳在未来真实窗口收集原字节，同时寻找**实质不同、决策前可用、能同时改变配置意图与自身风险容量**的新机制及匹配标签。来源日不计策略前向；即使新0bp账户历史通过，仍须2/5bp、风险和独立前向全部v2门。当前Goal未完成；本限定队列结论也不等于证明未来策略不可能。
